### News

Placeholder

### Installation

In [ ]:
# Placeholder

### Unsloth

Fine-tune **EmbeddingGemma 2** for sound to caption retrieval with LoRA and compare Recall@K before and after.

In [ ]:
from unsloth import FastSentenceTransformer
import torch

model = FastSentenceTransformer.from_pretrained(
    model_name = "unsloth/embeddinggemma-2",
    max_seq_length = 1024, # audio is 25 tokens per second: a 30 s clip is 750 tokens
    config_kwargs = {"vision_config": None}, # skip the vision tower: less VRAM
    full_finetuning = False,
)

In [ ]:
# @title Small display helpers (thumbnails, audio players, video)
import io, base64, html, torch
from IPython.display import display, HTML, Audio, Video
from PIL import Image

def show_images(images, captions = None, size = 160):
    cells = []
    for i, img in enumerate(images):
        buf = io.BytesIO(); img.copy().convert("RGB").resize((size, size)).save(buf, format = "JPEG")
        cap = html.escape(captions[i]) if captions else ""
        cells.append(f'<div style="display:inline-block;margin:4px;width:{size}px;font-size:11px;vertical-align:top">'
                     f'<img src="data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}"><br>{cap}</div>')
    display(HTML("".join(cells)))

def recall_at_k(similarity, positives, ks = (1, 5, 10)):
    """similarity: [queries, items]; positives[q] = set of correct item indices."""
    ranking = similarity.argsort(dim = 1, descending = True)
    out = {}
    for k in ks:
        top = ranking[:, :k].tolist()
        out[f"R@{k}"] = round(100 * sum(len(set(t) & positives[q]) > 0 for q, t in enumerate(top)) / len(top), 2)
    return out

<a name="Data"></a>
### Data Prep
Train on Clotho development (~300 clips), test on Clotho evaluation (209 clips).

In [ ]:
import random, soundfile as sf, numpy as np
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download, HfApi
from datasets import Dataset

import os
os.makedirs("clotho", exist_ok = True)
def save_clip(row): # write the wav once; sentence-transformers decodes + resamples audio files itself
    path = f"clotho/{row['index'].rsplit('/', 1)[-1]}.wav"
    open(path, "wb").write(row["audio"]["bytes"])
    return path

def split_captions(text):
    return [c.strip() + "." for c in text.split(". ") if c.strip()]

train_files = sorted(f.rfilename for f in HfApi().dataset_info("CLAPv2/Clotho").siblings if f.rfilename.startswith("data/train/"))[:10]
random.seed(3407)
pairs = []
for f in train_files:
    for r in pq.read_table(hf_hub_download("CLAPv2/Clotho", f, repo_type = "dataset")).to_pylist():
        clip = save_clip(r)
        for c in split_captions(r["text"]):
            pairs.append({"anchor": c, "positive": clip})
random.shuffle(pairs)
train_dataset = Dataset.from_list(pairs)

test_rows = pq.read_table(hf_hub_download("mteb/Clotho", "data/test-00000-of-00005.parquet", repo_type = "dataset")).to_pylist()
test_items = [save_clip(r) for r in test_rows]
test_texts, text_to_item = [], []
for i, r in enumerate(test_rows):
    for c in split_captions(r["text"]):
        test_texts.append(c); text_to_item.append(i)
print(f"train pairs: {len(train_dataset)}, test: {len(test_items)} clips / {len(test_texts)} captions")
print(pairs[0]["anchor"]); display(Audio(filename = pairs[0]["positive"]))

## Baseline Performance

In [ ]:
def evaluate(model):
    model.eval()
    with torch.no_grad():
        items = model.encode([{"audio": x} for x in test_items], batch_size = 8, convert_to_tensor = True)
        texts = model.encode(test_texts, batch_size = 64, convert_to_tensor = True)
    sim = model.similarity(texts, items).float().cpu()
    item_to_texts = [set(j for j, it in enumerate(text_to_item) if it == i) for i in range(len(test_items))]
    return {"text->audio": recall_at_k(sim, [{text_to_item[q]} for q in range(len(test_texts))]),
            "audio->text": recall_at_k(sim.T, item_to_texts)}

baseline = evaluate(model)
print(baseline)

We add LoRA adapters to the language model and the audio encoder (`finetune_audio_layers = True`).

In [ ]:
model = FastSentenceTransformer.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    finetune_audio_layers = True, # also adapt the audio encoder
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # saves VRAM for the audio tower
    random_state = 3407,
    task_type = "FEATURE_EXTRACTION",
)

<a name="Train"></a>
### Train the model
We train for 60 steps; set `num_train_epochs = 1` and `max_steps = -1` for a full run.

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from unsloth import is_bf16_supported

trainer = SentenceTransformerTrainer(
    model = model,
    train_dataset = train_dataset,
    loss = losses.MultipleNegativesRankingLoss(model),
    args = SentenceTransformerTrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 1,
        max_steps = 60,
        learning_rate = 1e-4,
        warmup_ratio = 0.05,
        lr_scheduler_type = "linear",
        logging_steps = 5,
        bf16 = is_bf16_supported(),
        fp16 = not is_bf16_supported(), # Unsloth switches EmbeddingGemma 2 to float32 math on T4
        remove_unused_columns = False,
        report_to = "none",
        output_dir = "outputs",
        save_strategy = "no",
        seed = 3407,
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"Peak reserved memory = {used_memory} GB of {max_memory} GB.")

### Evaluate after fine-tuning

In [ ]:
finetuned = evaluate(model)
for direction in baseline:
    print(f"{direction:12s} before {baseline[direction]}  after {finetuned[direction]}")

<a name="Inference"></a>
### Inference

In [ ]:
item_embeddings = model.encode([{"audio": x} for x in test_items], batch_size = 8, convert_to_tensor = True)
for query in ["birds chirping in a forest", "a car driving past on a wet road"]:
    scores = model.similarity(model.encode(query, convert_to_tensor = True), item_embeddings)[0]
    best = scores.argmax().item()
    print(f"{query} -> {test_texts[text_to_item.index(best)]} ({scores[best]:.3f})")
    display(Audio(filename = test_items[best]))

<a name="Save"></a>
### Saving
`save_pretrained` saves the LoRA adapters, `save_pretrained_merged` the full model.

In [ ]:
model.save_pretrained("lora_model")  # Local saving
model.tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("model", tokenizer = model.tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("hf/model", tokenizer = model.tokenizer, save_method = "merged_16bit", token = "")